> ## SOLUTION / ANSWER KEY &mdash; Lab 8.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-the-supervisor.ipynb`](../lab-02-the-supervisor.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.2 &mdash; The Supervisor (Router)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Express the supervisor as a StateGraph with conditional edges
- Route a two-intent ticket to BOTH real create_agent specialists
- Fall back to a general node when nothing matches

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The supervisor (router) pattern](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-08-02")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The **supervisor** is the backbone of a multi-agent system (deck slide 6): it reads the message and decides
**which specialist** handles it. In LangGraph you express this with a **`StateGraph`** and **conditional
edges** &mdash; `add_conditional_edges` sends control from the supervisor node to the specialist node(s) a
routing function names. Returning a **list** routes to **several** specialists at once (a billing *and* a
tech problem); an unmatched message falls back to a **general** node &mdash; the router never drops a ticket.
Each specialist node is a **real `create_agent`**, so the supervisor dispatches to agents that really answer,
and a **reducer** (`Annotated[list, add]`) gathers their findings into shared state.

In [ ]:
# supervisor node -> conditional edges -> real billing/tech specialist nodes (or general)
print("we build a real StateGraph: a supervisor routes each ticket to the specialist(s) it needs")

## Build it
Complete `supervise` (return the matching specialists, or `"general"`) and the **conditional edges** in
`build_team`. The specialist nodes are real `create_agent` agents, already wired for you.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

class TeamState(TypedDict):
    message: str
    findings: Annotated[list, add]        # reducer: each specialist APPENDS; nobody overwrites

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def specialist_node(role, tools):
    """Wrap a REAL create_agent specialist as a graph node that appends its finding."""
    agent = build_specialist(tools, role)
    def node(state):
        r = agent.invoke({"messages": [("user", state["message"])]}, config={"recursion_limit": 8})
        text = r["messages"][-1].content
        return {"findings": [f"{role}: {text}"]}
    return node

def general_node(state):
    return {"findings": ["general: forwarded to a human agent"]}

def supervise(state):
    """The supervisor: classify the message, name the specialist(s) it needs."""
    m = state["message"].lower()
    hits = []
    if any(k in m for k in ("charg", "refund", "invoice", "billed")): hits.append("billing")
    if any(k in m for k in ("crash", "bug", "login", "broken")): hits.append("tech")
    return hits or "general"

# Workflow (StateGraph) -- a supervisor conditional-routes to specialist node(s):
#
#                    START
#                      |
#                [ supervisor ]        (router: one, several, or general)
#                 /    |    \
#           billing   tech   general   (each a real create_agent)
#                 \    |    /
#                     END
def build_team():
    g = StateGraph(TeamState)
    g.add_node("supervisor", lambda s: s)                       # its only job is to route
    g.add_node("billing", specialist_node("billing", [lookup_invoice]))
    g.add_node("tech", specialist_node("tech", [known_issues]))
    g.add_node("general", general_node)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", supervise, ["billing", "tech", "general"])
    for n in ("billing", "tech", "general"):
        g.add_edge(n, END)
    return g.compile()

try:
    # The router itself is pure classification -- run it offline (no model call):
    for msg in ["I was charged twice", "the app keeps crashing",
                "charged twice and it crashes", "what are your hours?"]:
        print(f"{msg:34} -> {supervise({'message': msg})}")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the graph, then route a real two-intent ticket. The supervisor fans it to **both** real specialists; the reducer collects what each found.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        team = build_team()
        print("graph nodes:", sorted(set(team.get_graph().nodes) - {"__start__", "__end__"}))

        # A two-intent ticket: the supervisor routes it to BOTH real specialists, which really answer.
        state = team.invoke(
            {"message": "I was charged twice for order 4471 and the app keeps crashing on login.", "findings": []},
            config={"recursion_limit": 12})
        print("\nfindings gathered into shared state:")
        for f in state["findings"]:
            print(" -", f)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `add_conditional_edges` turns the supervisor into a real **router**: a two-intent ticket routes to **both** `billing` and `tech`, and the reducer merges their findings &mdash; **no join code needed**, the framework does it.
- Each specialist node is a real `create_agent` (a `CompiledStateGraph`); each calls **its own** tool to answer in its lane.
- An unmatched message routes to `general` &mdash; the safe fallback. In Lab 8.11 this same graph gains synthesis and a refund gate.

## Your turn (open task &mdash; no grader)
Add a third specialist &mdash; an **`account`** node (keywords `"password"`, `"email"`, `"cancel"`) with its own
`@tool` &mdash; to `build_team` and to `supervise`, then send it a matching ticket. **What good looks like:**
`add_conditional_edges` now reaches `account` too, a two-intent message still fans to both owners, and anything
unmatched still falls back to `general`. (Each live ticket makes a model call per engaged specialist &mdash; run a
couple on the free tier.)

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    if groq_ready():
        @tool
        def account_status(query: str) -> str:
            """Look up an account action such as a password reset or cancellation."""
            return "password reset link sent" if "password" in query.lower() else "account action logged"

        def supervise3(state):
            m = state["message"].lower(); hits = []
            if any(k in m for k in ("charg", "refund", "invoice", "billed")): hits.append("billing")
            if any(k in m for k in ("crash", "bug", "login", "broken")): hits.append("tech")
            if any(k in m for k in ("password", "email", "cancel")): hits.append("account")
            return hits or "general"

        g = StateGraph(TeamState)
        g.add_node("supervisor", lambda s: s)
        g.add_node("billing", specialist_node("billing", [lookup_invoice]))
        g.add_node("tech", specialist_node("tech", [known_issues]))
        g.add_node("account", specialist_node("account", [account_status]))
        g.add_node("general", general_node)
        g.add_edge(START, "supervisor")
        g.add_conditional_edges("supervisor", supervise3, ["billing", "tech", "account", "general"])
        for n in ("billing", "tech", "account", "general"): g.add_edge(n, END)
        team3 = g.compile()

        out = team3.invoke({"message": "please reset my password and refund the duplicate charge on 4471", "findings": []},
                           config={"recursion_limit": 12})
        for f in out["findings"]: print(" -", f)
    else:
        print("(add GROQ_API_KEY to .env)")
except Exception as e:
    print("(Live model hiccup -- a rate limit or a stray built-in tool call. Re-run in a moment.)", type(e).__name__)

---
### Key takeaway
The supervisor is a real StateGraph router: add_conditional_edges dispatches one ticket to one, several, or a fallback specialist -- each a real create_agent -- and a reducer gathers their findings. Next: the shared state that reducer writes to.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>